# The written explanations, beyond the word lists: Python steps

This notebook runs, in order, every Python step of the cross-study analyses of the written explanations. Each step is a script in its own folder, which this notebook calls exactly as the pipeline does, so the code lives in one place. After the R steps (`BeyondWordLists.Rmd`), every table the paper uses is up to date.

All of these analyses are **exploratory**: they were specified after every registered result was known, and none changes a registered verdict.

| Step | Folder | What it asks | Time |
|---|---|---|---|
| 1 | `../text_battery/` | What the explanations name (word lists), and who they say made the work (the agency rule) | ~12 min |
| 2 | `../image_grounding/` | How closely each explanation describes its own picture (CLIP; no word list) | ~20 min |
| 3 | `../label_from_meaning/` | Can the label be read from what people meant, beyond their rating? (sentence embeddings; no word list) | ~12 min once the embeddings are cached; about 45 more the first time |
| 4 | `../text_battery/origin_reason/` | Does an explanation give how the work was made as the *reason* for its rating? (grammar rule, validated against three model annotators) | ~2 min |
| 5 | `../typical_explanations/` | The most typical explanation of each group, chosen by a fixed rule, for quoting | ~1 min |

Run it with the `python312` kernel (Python 3.12). The times are those of a full test run on 6 November 2026 on a 12-core Mac, 46 minutes in all, with every results table identical to the one before. Each study's own pipeline must have been run first. Nothing here downloads anything: the models are read from the local Hugging Face cache (see each folder's README for the one-time fetch).

In [ ]:
import os, subprocess, sys
import pandas as pd

HERE = os.path.abspath(os.getcwd())
CROSS = os.path.dirname(HERE)  # the Cross-study folder
assert os.path.basename(HERE) == "beyond_word_lists", "Run this notebook from its own folder."


def run(folder, *args):
    """Run one script from its own folder with this notebook's Python, and show its output."""
    result = subprocess.run([sys.executable, *args], cwd=os.path.join(CROSS, folder),
                            capture_output=True, text=True)
    print(result.stdout[-3000:])
    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError(f"{args[0]} stopped; see the message above")


def table(folder, name):
    return pd.read_csv(os.path.join(CROSS, folder, "tables", name), keep_default_na=False)

## Step 1. The text battery: word lists and the agency rule

`TextBattery_01_Measure.py` counts every content word of every explanation and applies the frozen agency rule, reading only the participant, the work, the question and the text. Its second step, which joins the labels and ratings, is in R (`BeyondWordLists.Rmd`). The agency rule's validation is rescored from the saved annotations in a few seconds; it sends nothing.

In [ ]:
run("text_battery", "TextBattery_01_Measure.py")
run(os.path.join("text_battery", "agency"), "AgencyValidation.py", "score", "validation")
table(os.path.join("text_battery", "agency"), "agency_validation_scores.csv")

## Step 2. Image grounding (CLIP)

CLIP is a network trained to match pictures with their captions. Each explanation is treated as a caption, and its **grounding** is its similarity to its own picture minus its average similarity to the other nine artworks. This step is blind: it never reads the label or the rating. The R step fits the models.

In [ ]:
run("image_grounding", "ImageGrounding_01_Measure.py")

## Step 3. The label from meaning

Each explanation becomes a list of numbers from a sentence-embedding model, so that explanations that mean the same thing in different words lie close together. Per participant, a logistic regression predicts the label from the rating and explanation length alone, and then with the meaning added. The gain is tested against 100 null runs that keep what meaning shares with the rating and shuffle the rest. The same analysis runs without the explanations that mention prompting or a machine, and with two other embedding models (see the folder's README for the one-time download).

In [ ]:
run("label_from_meaning", "LabelFromMeaning.py")
lfm = table("label_from_meaning", "label_from_meaning.csv")
gain = lfm[lfm.quantity.str.startswith("AUC gain")].copy()
gain["shown"] = gain.estimate.astype(float) > gain.null_95th.astype(float)
gain[["model", "explanations", "study_paper", "measure", "estimate", "null_95th", "shown"]]

## Step 4. Origin as reason

A grammar rule flags explanations that give how the work was made (by AI, by prompting, or by hand) as a reason for the rating, following `origin_reason_codebook.md`. It was adjusted on a development sample of 250 explanations, frozen, and scored once on a fresh validation sample of 400 against three model annotators. The scores below come from the saved annotations; nothing is sent. Then the frozen rule codes every explanation, reading only the text.

In [ ]:
run(os.path.join("text_battery", "origin_reason"), "OriginReason.py", "score", "validation")
run(os.path.join("text_battery", "origin_reason"), "OriginReason.py", "apply")
table(os.path.join("text_battery", "origin_reason"), "origin_reason_validation_scores.csv")

## Step 5. Typical explanations

For each Part, question and label, the explanation whose meaning lies closest to the group's average (each participant counted once), among those at least as long as the group's median. The paper quotes these, so that the authors did not choose them.

In [ ]:
run("typical_explanations", "TypicalExplanations.py")
quotes = table("typical_explanations", "typical_explanations.csv")
quotes[["part", "question", "label", "rank", "study_paper", "rating", "text"]]

## Next

Knit `BeyondWordLists.Rmd` in this folder. It runs the R steps and shows the results.